# Kettős inga szimuláció – 4. rendű Runge–Kutta módszerrel 
Ez a projekt egy kettős inga (double pendulum) szimulációját valósítja meg negyedrendű Runge–Kutta (RK4) módszerrel. A rendszer több komponensből áll, amelyek különböző célokat szolgálnak, beleértve a szimulációt, adatkinyerést Python számára, valamint a nagyfelbontású heatmapeket (mert cpp, nem a felbontás miatt) GNUPLOT segítségével.

---

## Főbb komponensek

- `main.cpp` – A szimulációs program főfájlja (`my_program`), amely terminálról olvasott paraméterek alapján végzi a szimulációt és menti az eredményeket CSV-fájlba.
- `main2.cpp` – Dinamikus könyvtár (`libpendulum.dylib`), amely a szimulációs logikát Pythonból hívható módon teszi elérhetővé, például a kezdőszög–átfordulási idő összefüggésének vizsgálatához.
- `main3.cpp` – Egy segédprogram (`my_plotter`), amely a `main2.cpp`-vel készített dinamikus könyvtár használatához hasonló ábra készítéséhez hasznos, csak nagyobb felbontású GNUPLOT-rajzokat készít.

---

## Buildelés CMake segítségével

```bash
mkdir build
cd build
cmake ..
make


### Használati útmutató a `my_program` futtatásához

A `my_program` nevű szimulációs program **kötelezően** várja az összes bemeneti paramétert parancssori argumentumként. Nincs automatikusan beállított alapértelmezés, így minden futtatásnál meg kell adni a szükséges értékeket az alábbi sorrendben:

```
./my_program l1 l2 m1 m2 phi1_0 phi2_0 omega1_0 omega2_0 T h
```

#### Paraméterek jelentése:

- `l1`, `l2` – az első és második inga hossza
- `m1`, `m2` – az első és második inga tömege
- `phi1_0`, `phi2_0` – kezdeti szögek radiánban
- `omega1_0`, `omega2_0` – kezdeti szögsebességek [rad/s]
- `T` – szimuláció teljes időtartama [s]
- `h` – időlépés [s]

❗ **Figyelem:** ha bármelyik paraméter hiányzik vagy hibás, a program nem fog helyesen működni, hiszen a feladatleírás azt kérte adjuk megőket, nem akartam feleslegesen variálni.


### A `libpendulum.dylib` (vagy `libpendulum.so`, vagy `libpendulum.dll`) használata

A dinamikusan betölthető könyvtár (`libpendulum.dylib` macOS alatt, `libpendulum.so` Linux alatt, `libpendulum.dll` ha Windows) Pythonból hívható a `ctypes` segítségével. A könyvtár nem tartalmaz alapértelmezett értékeket: **minden szimulációs paramétert expliciten meg kell adni** a meghívás során.

A könyvtár fő célja:  
- egy adott kezdőszög esetén kiszámítani az inga időbeli szögfejlődését;
- megkeresni az első teljes átfordulás idejét.

#### Főbb függvények (például):

```python
lib.simulate_phi(l1, l2, m1, m2, phi1_0, phi2_0, omega1_0, omega2_0,
                 T, h, use_phi1,
                 phi_array_ptr, time_array_ptr, N)
```

- **Bemeneti paraméterek:**
  - `l1`, `l2` – inga hosszak (float)
  - `m1`, `m2` – tömegek (float)
  - `phi1_0`, `phi2_0` – kezdeti szögek (radiánban)
  - `omega1_0`, `omega2_0` – kezdeti szögsebességek
  - `T`, `h` – teljes idő, időlépés
  - `use_phi1` – ha 1, akkor $\phi_1$ figyelése; ha 0, akkor $\phi_2$
  - `phi_array_ptr` – egy előre allokált `double[]` tömb címe a szögeknek
  - `time_array_ptr` – időlépéseket tartalmazó tömb címe
  - `N` – lépések száma, általában: `int(T / h) + 1`
- **Visszatérés:** nincs (`void` típus) – az eredményeket a megadott tömbökön keresztül tölti fel

```python
lib.find_first_flip_time(time_array_ptr, phi_array_ptr, N)
```

- **Bemeneti paraméterek:**
  - `time_array_ptr` – időlépések tömbjének címe (`double*`)
  - `phi_array_ptr` – a kiválasztott szög értékei (`double*`)
  - `N` – a tömb hossza (egyezzen az `N = int(T / h) + 1` értékkel)
- **Visszatérés:** az első $(2k+1)\pi$ értéken való átfordulás ideje (float), vagy `-1.0`, ha nem történt átfordulás

#### Fontos:
- Használat előtt a könyvtárat `ctypes.CDLL()` vagy `ctypes.cdll.LoadLibrary()` segítségével kell betölteni.

❗ **Nincs automatikus alapértelmezés semmilyen paraméterre**, minden értéket explicit módon meg kell adni Pythonból!




A szimuláció kért ábrázolásait a `abrazolas.ipynb` illetve `abrazolas-tetszoleges.ipynb` notebookokkal végeztem el.

### A `my_plotter` (main3.cpp) program működése

A `main3.cpp` fájl a `my_plotter` nevű programot hozza létre, amely két dimenziós hőtérképet generál az első átfordulási időkről különböző kezdőszögek esetén. Az ábrák PNG formátumban kerülnek mentésre, GNUPLOT segítségével.

#### A program célja

A program célja, hogy $\varphi_1(0)$ és $\varphi_2(0)$ kezdőszögek egy rácsán végigsétálva:
- kiszámítsa az első inga (`use_phi1 = true`) és második inga (`use_phi1 = false`) első teljes átfordulásának idejét,
- ezeket az értékeket egy 2D mátrixba mentse,
- majd ezeket GNUPLOT segítségével hőtérképként ábrázolja.

#### A program működése

1. A $\varphi_1(0), \varphi_2(0) \in [-3.1, 3.1]$ intervallumot egy $1000 \times 1000$-as rácsban mintavételezi.
2. Minden rácspontban kétszer futtat szimulációt (`simulate_phi`):
   - egyszer $\varphi_1$ követésére,
   - egyszer $\varphi_2$ követésére.
3. A szimulált szögidőfüggvényből meghívja a `find_first_flip_time` függvényt.
4. Az így kapott átfordulási időket eltárolja két mátrixba.
5. A `plot_data()` függvénnyel ezekből PNG-formátumú GNUPLOT hőtérképeket generál:
   - `phi1.png` – $\varphi_1$ első átfordulási ideje,
   - `phi2.png` – $\varphi_2$ első átfordulási ideje.

#### Fájlok és kimenetek

- A program kimenete két kép:
  - `phi1.png`: $\varphi_1(0), \varphi_2(0) \mapsto$ átfordulási idő (ha $\varphi_1$-et követjük)
  - `phi2.png`: $\varphi_1(0), \varphi_2(0) \mapsto$ átfordulási idő (ha $\varphi_2$-t követjük)

#### Fontos technikai részletek

- A szimuláció fix paraméterekkel dolgozik, a fileban kell átírni:  
  $T = 30.0, \quad h = 0.01, \quad \text{illetve}  \quad h = 0.001 $ és 100-as felbontás mellett, $l_1 = l_2 = 1.0, \quad m_1 = m_2 = 1.0, \quad \omega_1(0) = \omega_2(0) = 0$,
- A GNUPLOT kimenet `splot ... with image` stílusban rajzol, szép színezett palettával.
- A képarány és tengelyarány fixált (`set size ratio -1`), így a rács torzításmentes.
- A program terminálon soronként visszajelzést ad az aktuális számítási sor állásáról.

❗ A GNUPLOT-nak telepítve kell lennie és elérhetőnek a PATH-ban. A képek PNG-formátumban a futtatás könyvtárába kerülnek.


### Alkalmazott szimulációs paraméterek

A szimuláció során végig az alábbi paramétereket alkalmaztuk:

- $T = 10.0-30.0-100.0$ &nbsp;&nbsp;&nbsp; *(teljes szimulációs idő, másodpercben, jelölöm mikor melyik)*
- $h = 0.001-0.01$ &nbsp;&nbsp;&nbsp; *(időlépés nagysága, jelölöm mikor melyik)*
- $N = \left\lfloor \frac{T}{h} \right\rfloor + 1 = 100001$ &nbsp;&nbsp;&nbsp; *(lépések száma)*
- $l_1 = 1$ m, $l_2 = 0.5$ m &nbsp;&nbsp;&nbsp; *(inga hosszak)*
- $m_1 = 1$ kg, $m_2 = 1$ kg &nbsp;&nbsp;&nbsp; *(tömegek)*
- $\omega_1(0) = 0$, $\omega_2(0) = 0$ &nbsp;&nbsp;&nbsp; *(kezdeti szögsebességek)*


(Kivéve amikor majd jelölöm hogy nem!)


A tapasztalat az volt hogy a szép, helyes szimuláció $h=0.001$ s-es időlépéssel kielégítő, nyilván ekkor a T-t érdemes limitálni. (Így általában a következő 3 diagrammot 0.001 s-es lépéssel készítettem, azonban a továbbiakban van amikor a futásidő miatt a 0.01 s volt érdemes választanom!)

### 1. Az inga pályája az \(x\)–\(y\) síkban
(T= 100.0 s, h = 0.001 s)

A szimuláció során kiszámítottuk az első és második inga végpontjának mozgását. Az alábbi ábra a két test által bejárt pályát ábrázolja az \(x\)–\(y\) síkban. A rendszer érzékeny a kezdeti feltételekre, a görbék (nagy szögkitérítéssel indítva) kaotikus viselkedést mutatnak.

Az alábbi három diagrammot (pálya + 2 fázistér) $\varphi_{1}(0)=\varphi_{1}(0)=3.14$ kezdőértékkel szimulálva kaptam!

<p style="text-align: center;">
  <img src="palya.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>Az inga végpontjainak pályái</em>
</p>


### 2. Fázistérdiagramok

<p style="text-align: center;">
  <img src="phase1.png" width="48%">
  <img src="phase2.png" width="48%">
</p>

<p style="text-align: center;">
  <em>Balra: az első inga fázistere (\(\varphi_1\), \(\omega_1\)). &nbsp;&nbsp;&nbsp; Jobbra: a második inga fázistere (\(\varphi_2\), \(\omega_2\)).</em>
</p>


### 3. Átfordulási idő a kezdőszög függvényében

Ahogy már írtam, amikor nem volt átfordulás (az szimuláció ideje során) a visszatérési érték az (első) átfordulásának idejére -1. Ezt az ábrázolásnál is meghagytam hogy jobban lehessen látni a trendet!


A kezdőszöget közösen növeljük, majd viszgáljuk mindkét (tömeg) inga első átfordulásának idejét!



<p style="text-align: center;">
  <img src="atf-1.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>
    Az első (\(\varphi_1\)) és második (\(\varphi_2\)) inga első átfordulási ideje a kezdeti szögtől függően.<br>
    A szimuláció során \(\omega_1(0) = \omega_2(0) = 0\), továbbá:<br>
    \(T = 100.0\), \(h = 0.001\),<br>
    \(m_1 = m_2 = 1.0\), \(l_1 = 1.0\), \(l_2 = 0.5\) (phi1 görbe) és (phi2 görbe).<br>
    0–3.1 közötti 1000 darab kezdeti szögkitéréssel.
  </em>
</p>




<p style="text-align: center;">
  <img src="atf-2.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>
    Az első (\(\varphi_1\)) és második (\(\varphi_2\)) inga első átfordulási ideje a kezdeti szögtől függően.<br>
    A szimuláció során \(\omega_1(0) = \omega_2(0) = 0\), továbbá:<br>
    \(T = 100.0\), \(h = 0.001\),<br>
    \(m_1 = m_2 = 1.0\), \(l_1 = 1.0\), \(l_2 = 0.5\) (phi1 görbe) és (phi2 görbe).<br>
    0–2\(\pi\) közötti 100 darab kezdeti szögkitéréssel.
  </em>
</p>






#### Láthatjuk hogy egy szögértékik (mind a két inga esetén, az első inga esetén kb. 2.2 rad, a második esetén 1.4 rad-ig nem fordul át, logikus, kell annyi helyzeti energia hogy ez letudjon zajlani, ezután a további energia növekedéssel mind a két inga esetén csökken, majd az első inga esetén növekedést látunk mivel a mozgás elején kis időre nem vagy keveset mozdul el, csak utána gyorsul be.)

Heatmapeket készítettem ha a két kezdeti szögkitérés eltér! Ekkor a színadja meg az első átfordulás idejét! (Ha fehér nem fordult át a szimuláció ideje alatt!)


<p style="text-align: center;">
  <img src="atf-1-2.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>
    Az első (\(\varphi_1\)) és második (\(\varphi_2\)) inga első átfordulási ideje a kezdeti szögtől függően.
    A kezdeti \(\omega_1(0) = \omega_2(0) = 0\) értékeket rögzítettük, a többi paraméter:
    \( T = 10.0 \), \( h = 0.01 \), \(l_1 = 1.0\), \(l_2 = 0.5\).
  </em>
</p>



<p style="text-align: center;">
  <img src="atf-2-2.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>
    Az első (\(\varphi_1\)) és második (\(\varphi_2\)) inga első átfordulási ideje a kezdeti szögtől függően.
    A kezdeti \(\omega_1(0) = \omega_2(0) = 0\) értékeket rögzítettük, a többi paraméter:
    \( T = 100.0 \), \( h = 0.01 \), \(l_1 = 1.0\), \(l_2 = 1.0\).
  </em>
</p>




Azért végeztem szimulációt egyenlő hosszakkal, mert utánanézve a témának a kettő irodalmat találtam:


Chen Joe. [*Chaos from Simplicity: An Introduction to the Double Pendulum*](chen_2008_report.pdf). 5 February 2008.

Heyl Jeremy S. [*The Double Pendulum Fractal*](Double.pdf). Department of Physics and Astronomy, University of British Columbia, 11 August 2008.


<p style="text-align: center;">
  <img src="im1.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>
   chen_2008_report.pdf, 3. oldal, Chaos from Simplicity: An Introduction to the Double Pendulum, a két inga átfordulása
  </em>
</p>


<p style="text-align: center;">
  <img src="im2.png" style="max-width: 80%;">
</p>

<p style="text-align: center;">
  <em>
   Double.pdf, 3. oldal, The Double Pendulum Fractal, a második inga átfordulása
  </em>
</p>


Hogy nagyobb felbontást is el tudjak érni, cpp-ban impementáltam a a fordulás keresést, gondolkodtam hogy érdemesebb-e a keresést belevenni, a szimulációba, azonban mivel (ahogy látható a kövtkező képeken) sok az olyan pont ahol nincs átfordulás, így végig kéne futnia a szimulációnak, így nem lehet annyi futási időt spórolni hogy sokkal jobb képet készítsek, vagy ezeket gyorsabban generáljam, így maradtam annál hogy a szimulálás után kerestem az átfordulás értékeit!



Itt a paraméterek esetén egyenlőek az ingahosszak!

<p>
  <img src="build/phi1-RES-100-T-30-h-001.png" width="48%">
  <img src="build/phi2-RES-100-T-30-h-001.png" width="48%">
</p>

<p style="text-align: center;">
  100x100-as felbontás, T=30, h=0.001
</p>


<p>
  <img src="build/phi1-RES-1000-T-30-h-01.png" width="48%">
  <img src="build/phi2-RES-1000-T-30-h-01.png" width="48%">
</p>

<p style="text-align: center;">
  1000x1000-as felbontás, T=30, h=0.01, (0-1000, -3.1, 3.1 rad között, ezt későn vettem észre, és kb. 15-20 percre mire újra lefut, így ezt már javítani nem tudtam)
</p>


## Így nagyjából sikerült rekreálni!

# És egy animáció:
(Ehhez már nem rakok kódot, generálás parancsa az adatokhoz: ./my_program 1 1 1 1 3.14 3.00 0 0 60 0.01)


Látszik hogy nem tökéletes, ugye ez annak az oka hogy nem a jó 0.001-es hanem 0.01-es lépést választottam ! (Az túl nagy lenne.)

In [53]:
from IPython.display import Video
Video("double_pendulum_animation.mp4", width=600)
